In [ ]:
import numpy as np

def scaled_dot_product_attention(q, k, v, blocked_mask=None):
    """
    q: [..., query_length, head_dim]
    k: [..., key_length, head_dim]
    v: [..., key_length, value_dim]

    blocked_mask:
        Boolean mask broadcastable to the score shape.
        True means the position is blocked.
    """

    # Compare every query with every key.
    scores = q @ np.swapaxes(k, -1, -2)

    # Scale by the per-head query/key dimension.
    scores = scores / np.sqrt(q.shape[-1])

    if blocked_mask is not None:
        # Turn the mask into a boolean array and broadcast it to the score shape.
        blocked_mask = np.asarray(blocked_mask, dtype=bool)
        blocked_mask = np.broadcast_to(blocked_mask, scores.shape)

        # Check invalid querys (all keys blocked).
        if np.any(np.all(blocked_mask, axis=-1)):
            raise ValueError("Each query must have at least one allowed key.")

        scores = np.where(blocked_mask, -np.inf, scores)

    # Stable softmax over key positions.
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(shifted)
    weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

    # Aggregate values.
    output = weights @ v

    return output, weights